# Simulated frauds baseline

In [33]:
from parameters import Parameters, ClassificationParameters, CardSimParameters
from datetime import timedelta


modification = False
anomaly = True
p = Parameters(
    cardsim=CardSimParameters.paper_params(modification),
    clf_params=ClassificationParameters.paper_params(with_anomaly=anomaly, with_modification=modification),
    seed=100
)

b = p.load_banksys()

In [34]:
b._fast_forward(b.current_time + timedelta(days=30), compute_features=True, show_progress=True)

2023-04-01: : 515192trx [00:24, 21392.47trx/s]                         


In [ ]:
import polars as pl
import numpy as np

trx = pl.DataFrame(b.training_features[1:], schema=b.schema)
predicted = b.clf.predict(trx)
labels = np.array(b.training_labels[1:])

In [ ]:
from typing import Literal

def predicted_by(*clfs: Literal["Statistical", "BRF", "Rules", "Anomaly", "*"]):
    reason = b.clf.get_details()
    selected = reason.select(*clfs)
    score = selected.sum_horizontal().to_numpy()
    return score > 0

In [50]:
from sklearn import metrics

for anomaly in (True, False):
    for brf in (True, False):
        for rules in (True, False):
            for statistical in (True, False):
                clfs = []
                if anomaly:
                    clfs.append("Anomaly")
                if brf:
                    clfs.append("BRF")
                if rules:
                    clfs.append("Rules")
                if statistical:
                    clfs.append("Statistical")
                if len(clfs) == 0:
                    continue
                
                predicted = predicted_by(*clfs)
                print(
                    f"Anomaly: {anomaly}, BRF: {brf}, Rules: {rules}, Statistical: {statistical} => "
                    f"F1: {metrics.f1_score(labels, predicted):.4f}, "
                    f"Precision: {metrics.precision_score(labels, predicted):.4f}, "
                    f"Recall: {metrics.recall_score(labels, predicted):.4f}, "
                    f"Accuracy: {metrics.accuracy_score(labels, predicted):.4f}"
                )
# predicted = predicted_by("Statistical", "BRF", "Rules", "Anomaly")

# print("F1:", metrics.f1_score(labels, predicted))
# print("Precision:", metrics.precision_score(labels, predicted))
# print("Recall:", metrics.recall_score(labels, predicted))
# print("Accuracy:", metrics.accuracy_score(labels, predicted))
# print(metrics.confusion_matrix(labels, predicted))

Anomaly: True, BRF: True, Rules: True, Statistical: True => F1: 0.1648, Precision: 0.0919, Recall: 0.7964, Accuracy: 0.9190
Anomaly: True, BRF: True, Rules: True, Statistical: False => F1: 0.1651, Precision: 0.0921, Recall: 0.7962, Accuracy: 0.9191
Anomaly: True, BRF: True, Rules: False, Statistical: True => F1: 0.1648, Precision: 0.0919, Recall: 0.7964, Accuracy: 0.9190
Anomaly: True, BRF: True, Rules: False, Statistical: False => F1: 0.1651, Precision: 0.0921, Recall: 0.7962, Accuracy: 0.9191
Anomaly: True, BRF: False, Rules: True, Statistical: True => F1: 0.0350, Precision: 0.0204, Recall: 0.1233, Accuracy: 0.9318
Anomaly: True, BRF: False, Rules: True, Statistical: False => F1: 0.0343, Precision: 0.0200, Recall: 0.1202, Accuracy: 0.9320
Anomaly: True, BRF: False, Rules: False, Statistical: True => F1: 0.0350, Precision: 0.0204, Recall: 0.1233, Accuracy: 0.9318
Anomaly: True, BRF: False, Rules: False, Statistical: False => F1: 0.0343, Precision: 0.0200, Recall: 0.1202, Accuracy: 0.9

/home/yann/projects/python/RL_Attack_September24/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
